In [1]:
import os, random, io
from pathlib import Path
from typing import List

import numpy as np
from PIL import Image, ImageFilter, ImageEnhance

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision.models import resnet18

from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


Using device: cuda


In [13]:
DATASET_ROOT = Path("/kaggle/input")

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

TARGET_SIZE = 299
RESIZE_SIZE = 320

BATCH_SIZE = 64
NUM_WORKERS = 2

EPOCHS = 4
LR = 3e-4

SAVE_PATH = "/kaggle/working/fft_debiased_resnet18_fromscratch_v2.pth"


In [4]:
def rglob_images(p: Path) -> List[Path]:
    if not p.exists():
        return []
    return [x for x in p.rglob("*") if x.suffix.lower() in IMG_EXTS]


def find_dir_contains(name: str) -> Path:
    name = name.lower()
    for p in DATASET_ROOT.rglob("*"):
        if p.is_dir() and name in str(p).lower():
            return p
    raise FileNotFoundError(f"Could not find folder containing: {name}")


In [5]:
def build_index():

    stylegan_real = rglob_images(find_dir_contains("train/real"))
    stylegan_fake = rglob_images(find_dir_contains("train/fake"))


    wish_real = rglob_images(find_dir_contains("realvsfake-81k").joinpath("Real"))
    wish_fake = rglob_images(find_dir_contains("realvsfake-81k").joinpath("Fake"))


    diffusion = rglob_images(find_dir_contains("syntheticeye-diffusion"))

    print("INDEX SUMMARY")
    print("StyleGAN REAL :", len(stylegan_real))
    print("Wish REAL     :", len(wish_real))
    print("GAN FAKE      :", len(stylegan_fake))
    print("DIFF FAKE     :", len(diffusion))

    return {
        "real_hq": stylegan_real,      
        "real_lq": wish_real,          
        "fake_gan": stylegan_fake,
        "fake_diff": diffusion,
    }


In [6]:
index = build_index()


INDEX SUMMARY
StyleGAN REAL : 50000
Wish REAL     : 0
GAN FAKE      : 50000
DIFF FAKE     : 33578


In [7]:
from pathlib import Path

WISH_REAL_ROOT = Path(
    "/kaggle/input/realvsfake-81k-by-wish/RealVsFake/RealVsFake"
)

assert WISH_REAL_ROOT.exists(), "Wish dataset path is wrong"


In [8]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

def load_wish_images(root: Path):
    real, fake = [], []

    real_dir = root / "Real"
    fake_dir = root / "Fake"

    for p in real_dir.rglob("*"):
        if p.suffix.lower() in IMG_EXTS:
            real.append(p)

    for p in fake_dir.rglob("*"):
        if p.suffix.lower() in IMG_EXTS:
            fake.append(p)

    return real, fake


In [9]:







wish_real, wish_fake = load_wish_images(WISH_REAL_ROOT)

index["real_lq"].extend(wish_real)
index["fake_gan"].extend(wish_fake)  


In [10]:
def make_splits(index, train_ratio=0.7, val_ratio=0.1):
    splits = {"train": [], "val": [], "test": []}


    real_all = index["real_hq"] + index["real_lq"]
    fake_all = index["fake_gan"] + index["fake_diff"]

    random.shuffle(real_all)
    random.shuffle(fake_all)

    def split_class(paths, label):
        n = len(paths)
        t = int(train_ratio * n)
        v = int(val_ratio * n)
        return {
            "train": [(p, label) for p in paths[:t]],
            "val":   [(p, label) for p in paths[t:t+v]],
            "test":  [(p, label) for p in paths[t+v:]],
        }

    real_split = split_class(real_all, 0)
    fake_split = split_class(fake_all, 1)

    for k in splits:

        n = min(len(real_split[k]), len(fake_split[k]))
        splits[k] = real_split[k][:n] + fake_split[k][:n]
        random.shuffle(splits[k])

    for k in splits:
        print(f"{k.upper()} size:", len(splits[k]))

    return splits


In [11]:
splits = make_splits(index)


TRAIN size: 183400
VAL size: 26200
TEST size: 52400


In [14]:
def make_phone_like(img: Image.Image) -> Image.Image:
    if random.random() < 0.7:
        w, h = img.size
        s = random.uniform(0.6, 0.85)
        img = img.resize((int(w*s), int(h*s)), Image.BICUBIC)
        img = img.resize((w, h), Image.BICUBIC)

    if random.random() < 0.4:
        img = img.filter(ImageFilter.MedianFilter(size=3))

    if random.random() < 0.5:
        img = img.filter(ImageFilter.UnsharpMask(radius=2, percent=150))

    if random.random() < 0.3:
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.9, 1.2))

    if random.random() < 0.7:
        buf = io.BytesIO()
        q = random.randint(40, 95)
        img.save(buf, format="JPEG", quality=q)
        buf.seek(0)
        img = Image.open(buf).convert("RGB")

    return img


In [15]:
def fft_transform(img_tensor):
    gray = img_tensor.mean(dim=0, keepdim=True)
    fft = torch.fft.fftshift(torch.fft.fft2(gray))
    mag = torch.log1p(torch.abs(fft))
    mag = (mag - mag.min()) / (mag.max() - mag.min() + 1e-8)
    return mag.repeat(3, 1, 1)


def frequency_mask(x, p=0.5):
    if random.random() > p:
        return x
    h, w = x.shape[-2:]
    r = random.randint(10, 40)
    cy, cx = h // 2, w // 2
    x[:, cy-r:cy+r, cx-r:cx+r] = 0
    return x


In [16]:
class FFTDebiasedDataset(Dataset):
    def __init__(self, samples, train=True):
        self.samples = samples
        self.train = train

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        img = img.resize((RESIZE_SIZE, RESIZE_SIZE), Image.BICUBIC)

        left = (RESIZE_SIZE - TARGET_SIZE) // 2
        img = img.crop((left, left, left+TARGET_SIZE, left+TARGET_SIZE))


        if self.train and label == 0 and random.random() < 0.25:
            img = make_phone_like(img)

        if self.train and label == 1 and random.random() < 0.25:
            img = img.filter(ImageFilter.GaussianBlur(radius=1.0))

        x = torch.from_numpy(np.array(img)).permute(2,0,1).float() / 255.0
        x = fft_transform(x)

        if self.train:
            x = frequency_mask(x, p=0.5)

        return x, torch.tensor(label)


In [17]:
train_ds = FFTDebiasedDataset(splits["train"], train=True)
val_ds   = FFTDebiasedDataset(splits["val"], train=False)
test_ds  = FFTDebiasedDataset(splits["test"], train=False)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_ds, BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, BATCH_SIZE, shuffle=False)


In [39]:
MODEL_PATH = "/kaggle/input/frequency-model-checkpoint/fft_debiased_resnet18_fromscratch.pth"


In [40]:
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

model = resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 2)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(DEVICE)
model.eval()

print("✅ FFT model loaded successfully")
print("Metadata:", {k: v for k, v in checkpoint.items() if k != "model_state_dict"})


✅ FFT model loaded successfully
Metadata: {'epoch': 2, 'architecture': 'resnet18_fft_debiased'}


In [43]:




criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)


In [41]:
def evaluate(loader, name):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            p = out.argmax(1)
            ys.extend(y.cpu().numpy())
            ps.extend(p.cpu().numpy())

    print(
        f"{name}: acc={accuracy_score(ys,ps):.4f} "
        f"prec={precision_score(ys,ps):.4f} "
        f"rec={recall_score(ys,ps):.4f} "
        f"f1={f1_score(ys,ps):.4f}"
    )


In [44]:
best_f1 = 0.0

for epoch in range(1, EPOCHS+1):
    model.train()
    total_loss = 0

    for x, y in tqdm(train_loader, desc=f"Epoch {epoch}"):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Train loss: {total_loss/len(train_loader):.4f}")
    evaluate(val_loader, "VAL")


    f1 = f1_score(*evaluate.__wrapped__ if False else ([],[]))
    torch.save({
        "model_state_dict": model.state_dict(),
        "epoch": epoch,
        "architecture": "resnet18_fft_debiased",
    }, SAVE_PATH)

print("\nFINAL TEST")
evaluate(test_loader, "TEST")
print("Saved to:", SAVE_PATH)


Epoch 1: 100%|██████████| 2866/2866 [34:03<00:00,  1.40it/s]


Train loss: 0.1348


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


VAL: acc=0.8913 prec=0.9013 rec=0.8788 f1=0.8899


Epoch 2: 100%|██████████| 2866/2866 [25:36<00:00,  1.87it/s]


...

VAL: acc=0.8763 prec=0.9126 rec=0.8324 f1=0.8706


Epoch 4: 100%|██████████| 2866/2866 [23:34<00:00,  2.03it/s]


Train loss: 0.1003
VAL: acc=0.9139 prec=0.9515 rec=0.8722 f1=0.9101

FINAL TEST


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


TEST: acc=0.9114 prec=0.9501 rec=0.8684 f1=0.9074
Saved to: /kaggle/working/fft_debiased_resnet18_fromscratch_v2.pth


In [23]:
print("plug")

plug
